In [ ]:
%load_ext autoreload
%autoreload 2

# MCAP

In [ ]:
from pathlib import Path

from psilia import get_console
from psilia.data.mcap import (
    get_mcap_overview, 
    get_mcaps, 
    get_summary,
    get_topics,
    get_schemas,
    get_channel_overview,
    read_nth_message,
    parse_msg,
)
from psilia.data.lance import convert
from pandas import DataFrame
from psilia.data.lance import convert
from psilia.data.utils import save_yaml, load_yaml, psi_glob


console = get_console()

In [ ]:
console = get_console()
print = console.print

path = str(Path('~/workspace/data/rosbags/**/*.mcap').expanduser())
mcaps = psi_glob(path)
console.print(mcaps[["name", "path"]])

mcap = mcaps["path"][1]
overview = get_mcap_overview(mcap)
console.print(overview)

In [ ]:
save_yaml(
    [row.to_dict() for _,row in overview.iterrows()],
    "./_overview.yaml", 
)
console.print(
    DataFrame.from_dict(load_yaml("./_overview.yaml")))

In [ ]:
msg = read_nth_message(**overview.iloc[2])
parse_msg(msg)

In [ ]:
console.print(mcap)
console.print(mcap.parent)

In [ ]:
convert(mcap, target_dir= mcap.parent / "lance", mode="create")

In [ ]:
import time
from datetime import datetime

console.print(f"time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
type(time.strftime('%Y-%m-%d %H:%M:%S'))
time.time()
str(datetime.now())

In [ ]:
load_yaml("./_lance/overview.yaml")

In [ ]:
from psilia.data.utils import np_save_bytes, np_load_bytes
import lance

ds = lance.LanceDataset('./_lance/zed-zed_node-pose.lance')
console.print(ds.schema.names)

In [ ]:
m = read_nth_message(mcap, "/zed/zed_node/pose")
m.channel, m.schema

List camera info messages:

In [ ]:
overview = get_mcap_overview(
    mcap,
    filter_by_schema='sensor_msgs/msg/CameraInfo'
)
console.print(overview)

In [ ]:
from psilia.vision import CameraIntrinsics
from psilia.data.mcap import parse_msg


msg = read_nth_message(
    overview.attrs["mcap"],
    overview.iloc[0]["topic"])


console.print(CameraIntrinsics.from_camera_info_msg(msg))
console.print(CameraIntrinsics.from_camera_info_dict(parse_msg(msg)))

List different Pose and Transform messages

In [ ]:
overview = get_mcap_overview(
    mcap,
    filter_by_schema=[
         'tf2_msgs/msg/TFMessage',
         'geometry_msgs/msg/PoseStamped',
         'geometry_msgs/msg/Pose',
         'geometry_msgs/msg/TransformStamped',
         'geometry_msgs/msg/Transform',
    ]
)
console.print(overview)


```yaml
TFMessage:
    transforms: list[TransformStamped]

TransformStamped:
    header: Header
        stamp: Time
        frame_id: str
    child_frame_id: str
    transform: Transform

Transform:
    translation: Vector
    rotation: Quaternion

PoseStamped:
    header: Header
        stamp: Time
        frame_id: str
    pose: Pose

Pose:
    position: Point
    orientation: Quaternion

```

In [ ]:
from psilia.data.mcap import parse_msg
import matplotlib.pyplot as plt

mcap = mcaps["path"][0]
overview = get_mcap_overview(mcap)
console.print(overview)


msg1 = read_nth_message(
    mcap,
    "/zed/zed_node/left/image_rect_color")

msg2 = read_nth_message(
    mcap,
    "/zed/zed_node/depth/depth_registered")


fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(parse_msg(msg1)["data"])
axs[1].imshow(parse_msg(msg2)["data"])
